# 01 — EDA Open Bandit Dataset


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.dataset import OpenBanditDataset

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)


In [2]:
POLICY_RANDOM = "random"
POLICY_BTS    = "bts"
CAMPAIGN      = "all"


In [ ]:
def load_obd_feedback(behavior_policy: str, campaign: str = CAMPAIGN):
    dataset  = OpenBanditDataset(behavior_policy=behavior_policy, campaign=campaign)
    feedback = dataset.obtain_batch_bandit_feedback()
    return dataset, feedback


dataset_random, feedback_random = load_obd_feedback(POLICY_RANDOM)
dataset_bts,    feedback_bts    = load_obd_feedback(POLICY_BTS)

print(f"[{POLICY_RANDOM}] n_rounds:", feedback_random["n_rounds"])
print(f"[{POLICY_BTS}]   n_rounds:", feedback_bts["n_rounds"])


In [ ]:
def inspect_feedback(name: str, dataset: OpenBanditDataset, feedback: dict):
    print(f"\n{'='*55}")
    print(f"  Polityka: {name}")
    print(f"{'='*55}")
    print(f"  Klucze feedback : {list(feedback.keys())}")
    print(f"  n_rounds        : {feedback['n_rounds']}")
    print(f"  context shape   : {feedback['context'].shape}")
    print(f"  action_context  : {feedback['action_context'].shape}")
    print(f"  n_actions       : {dataset.n_actions}")
    print(f"  len_list        : {dataset.len_list}")

    action = feedback["action"]
    reward = feedback["reward"]
    print(f"\n  Unikalne akcje        : {len(set(action))}")
    print(f"  Proporcja reward=1 (CTR): {reward.mean():.4f}  ({reward.mean()*100:.2f}%)")

    print(f"\n  Pierwsze 5 wierszy (action / reward / pscore):")
    display(pd.DataFrame({
        "action": action[:5],
        "reward": reward[:5],
        "pscore": feedback["pscore"][:5],
    }))


inspect_feedback(POLICY_RANDOM, dataset_random, feedback_random)
inspect_feedback(POLICY_BTS,    dataset_bts,    feedback_bts)


In [ ]:
def summarize_feedback(name: str, feedback: dict) -> dict:
    action  = np.asarray(feedback["action"])
    reward  = np.asarray(feedback["reward"])
    context = np.asarray(feedback["context"])
    return {
        "policy":             name,
        "n_rounds":           int(feedback["n_rounds"]),
        "n_actions_observed": int(np.unique(action).shape[0]),
        "reward_mean (CTR)":  float(reward.mean()),
        "reward_std":         float(reward.std()),
        "context_shape":      tuple(context.shape),
        "missing_pscore":     int(np.isnan(feedback["pscore"]).sum()),
        "missing_reward":     int(np.isnan(reward).sum()) if np.issubdtype(reward.dtype, np.floating) else 0,
    }


summary_df = pd.DataFrame([
    summarize_feedback(POLICY_RANDOM, feedback_random),
    summarize_feedback(POLICY_BTS,    feedback_bts),
])
summary_df


## EDA


In [ ]:
def plot_reward_distribution(fb_random: dict, fb_bts: dict):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

    for ax, fb, name, color in zip(
        axes,
        [fb_random, fb_bts],
        [POLICY_RANDOM, POLICY_BTS],
        ["#1f77b4", "#ff7f0e"],
    ):
        counts = pd.Series(fb["reward"]).value_counts(normalize=True).sort_index()
        ax.bar(counts.index.astype(str), counts.values * 100, color=color, alpha=0.85)
        ax.set_title(f"Rozklad reward - {name}")
        ax.set_xlabel("reward")
        ax.set_ylabel("% obserwacji")
        for i, v in enumerate(counts.values):
            ax.text(i, v * 100 + 0.3, f"{v*100:.1f}%", ha="center", fontsize=10)

    plt.suptitle("Rozklad nagrod (reward) - porownanie polityk", fontweight="bold")
    plt.tight_layout()
    plt.savefig("../figures/reward_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()


plot_reward_distribution(feedback_random, feedback_bts)


In [ ]:
def plot_action_distribution(fb_random: dict, fb_bts: dict):
    a_rand = pd.Series(fb_random["action"]).value_counts(normalize=True).sort_index()
    a_bts  = pd.Series(fb_bts["action"]).value_counts(normalize=True).sort_index()

    all_actions = sorted(set(a_rand.index) | set(a_bts.index))
    a_rand  = a_rand.reindex(all_actions, fill_value=0)
    a_bts   = a_bts.reindex(all_actions, fill_value=0)
    uniform = 100 / len(all_actions)

    fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
    for ax, series, name, color in zip(
        axes, [a_rand, a_bts], [POLICY_RANDOM, POLICY_BTS], ["#1f77b4", "#ff7f0e"]
    ):
        ax.bar(all_actions, series.values * 100, color=color, alpha=0.8)
        ax.axhline(uniform, color="red", ls="--", lw=1.2,
                   label=f"Rownomierny ({uniform:.2f}%)")
        ax.set_title(f"Rozklad akcji - {name}")
        ax.set_ylabel("% wyboru")
        ax.legend()

    axes[1].set_xlabel("action id")
    plt.suptitle("Rozklad akcji - czy polityki sa rownomierne?", fontweight="bold")
    plt.tight_layout()
    plt.savefig("../figures/action_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()

    return pd.DataFrame({"random": a_rand, "bts": a_bts})


action_distribution = plot_action_distribution(feedback_random, feedback_bts)
action_distribution.describe()


In [ ]:
def compute_action_ctr(feedback: dict) -> pd.DataFrame:
    df = pd.DataFrame({"action": feedback["action"], "reward": feedback["reward"]})
    return df.groupby("action")["reward"].agg(ctr="mean", n_obs="count")


ctr_random = compute_action_ctr(feedback_random)
ctr_bts    = compute_action_ctr(feedback_bts)

mean_ctr_random = feedback_random["reward"].mean()
mean_ctr_bts    = feedback_bts["reward"].mean()

fig, axes = plt.subplots(2, 1, figsize=(16, 9))

for ax, ctr_df, mean_ctr, name, color in zip(
    axes,
    [ctr_random, ctr_bts],
    [mean_ctr_random, mean_ctr_bts],
    [POLICY_RANDOM, POLICY_BTS],
    ["#1f77b4", "#ff7f0e"],
):
    ax.bar(ctr_df.index, ctr_df["ctr"], color=color, alpha=0.75,
           label="P(reward=1|action)")
    ax.axhline(mean_ctr, color="red", linestyle="--", linewidth=1.8,
               label=f"Sredni CTR = {mean_ctr:.4f}")

    for action_id, val in ctr_df["ctr"].nlargest(3).items():
        n = int(ctr_df.loc[action_id, "n_obs"])
        ax.annotate(
            f"a{action_id}\nn={n}",
            xy=(action_id, val), xytext=(0, 6),
            textcoords="offset points",
            ha="center", fontsize=7, color="darkred",
        )

    ax.set_title(f"P(reward=1|action) - {name}  |  obserwowanych akcji = {len(ctr_df)}")
    ax.set_xlabel("action id")
    ax.set_ylabel("P(reward=1 | action)")
    ax.legend(fontsize=9)

fig.suptitle(
    "Naive comparison: P(reward=1|action) - selection bias visible",
    fontsize=14, fontweight="bold",
)
plt.tight_layout()
plt.savefig("../figures/selection_bias.png", dpi=150, bbox_inches="tight")
plt.show()

top_action = int(ctr_bts["ctr"].idxmax())
print(f"Sredni CTR - {POLICY_RANDOM}: {mean_ctr_random:.4f}")
print(f"Sredni CTR - {POLICY_BTS}:    {mean_ctr_bts:.4f}")
print(f"Najlepsza akcja wg {POLICY_BTS}: action={top_action}",
      f" CTR={ctr_bts.loc[top_action,'ctr']:.4f}",
      f" n_obs={int(ctr_bts.loc[top_action,'n_obs'])}")
